# Formatting Anki Flashcards

## Setup

In this notebook, we highlight some deficiencies of Qwen 3.5 when used in combination with the Completion API. These issues have contributed to the decision of using the more modern ChatCompletion API for our work in this addon.

For this notebook, we are going to use [`unsloth/Qwen3.5-27B-GGUF`](https://huggingface.co/unsloth/Qwen3.5-27B-GGUF). We selected `Qwen3.5-27B` because it's a good-performing open-weight LLM that fits in 24GB of VRAM and delivers acceptable inference speed.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from anki.collection import Collection

from addon.infrastructure.configuration.settings import AddonConfig

In [3]:
# Open an existing collection
col = Collection("/home/gianluca/.local/share/Anki2/User 1/collection.anki2")

# Do something with the collection
print(f"Number of notes: {col.note_count()}")
print(f"Number of cards: {col.card_count()}")

Number of notes: 3261
Number of cards: 3380


### Connect to Inference Server

In [4]:
from IPython.display import Markdown, display

from addon.domain.entities.note import AddonNote
from addon.infrastructure.llm.schemas import AddonNoteChanges


def display_addon_note(addon_note: AddonNote | AddonNoteChanges | str) -> None:
    if isinstance(addon_note, str):
        print(f"Content: {addon_note}")
    elif isinstance(addon_note, (AddonNote, AddonNoteChanges)):
        display(
            Markdown(
                f"**Front:** {addon_note.front}\n\n"
                f"**Back:** {addon_note.back}\n\n"
                f"**Tags:** {addon_note.tags}"
            )
        )
    else:
        raise ValueError(
            f"Expected str, AddonNote, or AddonNoteChanges, found {type(addon_note)}"
        )

In [5]:
from addon.infrastructure.external_services.openai import OpenAIClient


def answer(input: str | list[dict], guided_json=None, **kwargs):
    """Helper function to prompt LLM.

    input: Either a string (completions API) or list of message dicts (chat completion)
    kwargs: Can override config values like max_tokens, temperature, etc.
            extra_body: dict whose contents are merged into the request payload.
    """
    extra_body = kwargs.pop("extra_body", {})
    config = AddonConfig.create_nullable(kwargs)
    client = OpenAIClient.create(config)

    run_kwargs = {**extra_body}
    if guided_json is not None:
        run_kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "addon_note_changes",
                "schema": guided_json,
            },
        }

    content = client.run(input, **run_kwargs)
    reasoning_content = client.last_reasoning_content
    cleaned_content = content.replace("<think>\n\n</think>\n\n", "")

    if reasoning_content:
        print("=== Thinking ===")
        print(reasoning_content)
        print("=== End Thinking ===\n")

    display_addon_note(cleaned_content)

    if guided_json is not None:
        suggested_changes = AddonNoteChanges.model_validate_json(
            cleaned_content
        )
        return (suggested_changes, reasoning_content)

    return (cleaned_content, reasoning_content)

## Completions vs. Chat Completions API

In [6]:
prompt = "Respond only with one word, lowercase, without punctuation. What is the Italian word for 'hello'?"

In [7]:
for _ in range(3):
    content, reasoning_content = answer(
        input=prompt,
        mode="v1/completions",
        max_tokens=50,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print("\n######\n")

Content: <think>
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Translate the English word "hello" into Italian.
    *   Constraint 1: Respond only with one word.
    *

######

Content: <think>
Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What is the Italian word for 'hello'?"
    *   Constraint 1: Respond only with one word.
    *

######

Content: ciao

######



In [8]:
for _ in range(3):
    content, reasoning_content = answer(
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        mode="v1/chat/completions",
        max_tokens=50,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print("\n######\n")

Content: ciao

######

Content: ciao

######

Content: ciao

######



It seems that when using the Completion API, we cannot fully disable thinking.